# Post-Tune Analysis

This notebook generates post-tune result numbers. The computation itself lives in importable modules under `src/analysis/utils/` and the cells below only orchestrate them and display the output.

**Sections**

1. **Combined-intervention composition**: decides per heterogeneous environment which single interventions to carry into the combined run.

## 1. Combined-intervention composition

Decides per heterogeneous environment which single interventions to carry into the combined run.

**Mechanism.** Start from the union of all single interventions and drop one only when it *both* meaningfully and significantly harms the reference policy:

1. **Meaningful**: the seed-paired mean difference in normalized score is below `-0.02` (a loss worth more than 2% of the Random -> BS-Optimized span).
2. **Significant**: the 95% paired confidence interval lies entirely below zero.

Failing either test keeps the intervention, so the rule is deliberately conservative.

**Reference and pairing.** Each intervention is compared against the retuned MAPPO-PS baseline on the *same* 5 seeds (100-500). The paired comparison thus restricts to the common 5 seeds.

**Normalized score.** Per-seed returns are placed on a common axis by `(value - Random) / (BS-Optimized - Random)` for cross-environment comparability. BS-Optimized is chosen to be the upper anchor in every environment. 

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve().parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.analysis.utils.paired_stats import (
    normalized_scores,
    paired_difference,
)
from src.analysis.utils.summary_loader import load_seed_values
from src.analysis.utils.intervention_composition import (
    DEFAULT_DROP_THRESHOLD,
    REFERENCE_FILENAME,
    compose_environment,
    discover_intervention_files,
    drop_decision,
    format_environment_table,
    load_anchors,
    save_composition,
)

# Define directories, environments, and values
ANALYSIS_DIR = REPO_ROOT / "experiment_outputs" / "analysis_outputs"
RESULTS_DIR = REPO_ROOT / "experiment_outputs" / "analysis_results"
ENVIRONMENTS = ["env_demand_hetero", "env_cost_hetero", "env_combined_hetero"]
DROP_THRESHOLD = DEFAULT_DROP_THRESHOLD

# Print for check
print(f"Drop threshold (normalized score): {DROP_THRESHOLD}")
for env in ENVIRONMENTS:
    print(f"  {env}")

### 1.1 Inputs for one environment

Each environment directory holds the seed-evaluation summaries for the retuned shared-parameter MAPPO reference, for each intervention, and for the heuristic baselines.

In [3]:
# Get the intervention files
env_dir = ANALYSIS_DIR / ENVIRONMENTS[0]
intervention_files = discover_intervention_files(env_dir)

# Print for check
print(f"Environment: {env_dir.name}")
print(f"Reference:   {REFERENCE_FILENAME}")
print("Interventions:")
for label, path in intervention_files.items():
    print(f"  {label:<6} {path.name}")

Environment: env_demand_hetero
Reference:   baseline_mappo_ps.yaml
Interventions:
  A1     A1.yaml
  A2     A2.yaml
  A3     A3.yaml
  A4     A4.yaml


### 1.2 Anchors and normalized score

The two heuristics Random and BS-Optimized define the normalized-score range (Random at 0, BS-Optimized at 1). The `-0.02` margin is thus 2% of the Random - BS-Optimized span, which rescales automatically with each environment's difficulty.

In [6]:
# Get the anchor values and the span
anchors = load_anchors(env_dir)
span = anchors["upper"] - anchors["lower"]

# Print for check
print(f"Lower anchor (Random):       {anchors['lower']:>12.4f}")
print(f"Upper anchor (BS-Optimized): {anchors['upper']:>12.4f}")
print(f"Span:                        {span:>12.4f}")
print(f"-0.02 normalized == {DROP_THRESHOLD * span:.4f} raw return")

Lower anchor (Random):         -3588.6432
Upper anchor (BS-Optimized):    -410.8991
Span:                           3177.7441
-0.02 normalized == -63.5549 raw return


### 1.3 Seed-paired comparison

Take one intervention, normalize it and the reference, restrict to the seeds they share, and read off the paired difference, interval, and effect size. Every value in the table further down is produced exactly this way.

In [ ]:
# Get the normalized scores for the MAPPO reference
reference = normalized_scores(
    load_seed_values(env_dir / REFERENCE_FILENAME),
    anchors["lower"],
    anchors["upper"],
)

# Get the normalized scores for the intervention
label, path = next(iter(intervention_files.items()))
condition = normalized_scores(
    load_seed_values(path), anchors["lower"], anchors["upper"]
)

# Compute the paired difference between each intervention and the reference
paired = paired_difference(condition, reference)

# Print results
print(f"Intervention {label} vs reference on common seeds {paired['seeds']}")
print()
print("per-seed normalized difference (intervention - reference):")
for seed, diff in paired["per_seed_diff"].items():
    print(f"  seed {seed}: {diff:+.4f}")
print()
print(f"delta     = {paired['delta']:+.4f}")
print(f"95% CI    = [{paired['ci'][0]:+.4f}, {paired['ci'][1]:+.4f}]")
print(f"t_stat    = {paired['t_stat']:+.3f}")
print(f"p_value   = {paired['p_value']:.4f}")
print(f"cohen_dz  = {paired['cohen_dz']:+.3f}")
print(f"decision  = {drop_decision(paired, DROP_THRESHOLD)}")

Intervention A1 vs reference on common seeds [100, 200, 300, 400, 500]

per-seed normalized difference (intervention - reference):
  seed 100: -0.0000
  seed 200: -0.0006
  seed 300: +0.0002
  seed 400: +0.0054
  seed 500: +0.0040

delta     = +0.0018
95% CI    = [-0.0016, +0.0052]
t_stat    = +1.489
p_value   = 0.2106
cohen_dz  = +0.666
decision  = {'meaningful': False, 'significant': False, 'drop': False}


### 1.4 Composition table per environment

`compose_environment` repeats the comparison above for every intervention and applies the drop rule. `delta` and the interval are in normalized-score units, `d_z` is the standardized paired effect size. A `DROP` verdict means both conditions fired.

In [ ]:
# Compute paired-differences for each intervention and environment
results = [
    compose_environment(ANALYSIS_DIR / env, DROP_THRESHOLD)
    for env in ENVIRONMENTS
]

# Print results
for result in results:
    print(format_environment_table(result))
    print()

Environment: env_demand_hetero
Anchors: Random=-3588.6432  BS-Optimized=-410.8991
Label              delta                  95% CI      d_z   n  Verdict
----------------------------------------------------------------------
A1               +0.0018      [-0.0016, +0.0052]    +0.67   5  keep
A2               +0.0033      [+0.0005, +0.0061]    +1.47   5  keep
A3               -0.0049      [-0.0096, -0.0003]    -1.31   5  keep
A4               -0.0095      [-0.0192, +0.0001]    -1.23   5  keep
Retained: A1, A2, A3, A4  |  Dropped: none

Environment: env_cost_hetero
Anchors: Random=-1472.7804  BS-Optimized=-424.7367
Label              delta                  95% CI      d_z   n  Verdict
----------------------------------------------------------------------
A1               -0.0010      [-0.0086, +0.0067]    -0.16   5  keep
A2               +0.0043      [-0.0025, +0.0111]    +0.79   5  keep
A3               -0.0165      [-0.0203, -0.0127]    -5.43   5  keep
A4               -0.0322      [-0.

### 1.5 Final combined-intervention sets

Determine the final retained set per environment that the combined runs should turn on.

In [ ]:
# Build the final retained set
for result in results:
    retained = ", ".join(result["retained"]) or "none"
    dropped = ", ".join(result["dropped"]) or "none"
    print(f"{result['env']:<22}  retain: {retained:<18}  drop: {dropped}")

env_demand_hetero       retain: A1, A2, A3, A4      drop: none
env_cost_hetero         retain: A1, A2, A3          drop: A4
env_combined_hetero     retain: A1, A2, A3, A4      drop: none
